In [ ]:
'''
KODE-X — DAY 12 EXERCISE
-------------------------------------
TOPICS COVERED TILL DATE
--------------------------------------

PYTHON
- Variables, Data Types, Type Casting
- Operators, Control Flow (if/elif/else, loops)
- Strings, Lists, Tuples, Sets, Dictionaries
- Functions, *args, **kwargs, Lambda
- File Handling : CSV, JSON, Parquet
- Exception Handling, Logging

LINUX
- Topics 1, 2, 3, 4, 7 from HTML

SQL
- SELECT, WHERE, NULL Handling, ORDER BY
- DISTINCT, ALIAS, LIMIT
- Aggregate Functions : COUNT, SUM, AVG, MIN, MAX
- GROUP BY
- HAVING (new today)
- JOINS : INNER, LEFT, RIGHT, FULL OUTER, CROSS (new today)

PYSPARK
- SparkSession, Architecture, Lazy Evaluation
- DataFrame, Schema, Transformations vs Actions
- Column Operations : select, withColumn, alias, cast, lit
- Filtering : where, filter, isin, between, like, when
- NULL Handling : isNull, isNotNull, dropna, fillna
- Aggregations : groupBy, agg, count, sum, avg, min, max
- Joins : inner, left, right, full outer, left semi, left anti, broadcast
- String Functions : upper, lower, trim, concat, substring, length
- Date Functions : to_date, date_add, datediff, year, month, day
- Numeric Functions : round, ceil, floor, abs, sqrt (new today)

--------------------------------------------------
PART A — SQL : HAVING CLAUSE
--------------------------------------------------

Platform : freesql.com
Use the tables set up on Day 9
(employees, orders, products, customers,
 patients, doctors, appointments)

KEY RULE:
WHERE  — filters rows BEFORE grouping
HAVING — filters groups AFTER aggregation
You cannot use aggregate functions inside WHERE.
Use HAVING for that.

--------------------------------------------------

Q1. Basic HAVING

From employees table, find all departments
where the average salary is greater than 70000.

Show : department, avg_salary
Round avg_salary to 2 decimal places.
Order by avg_salary descending.

--------------------------------------------------

Q2. HAVING with COUNT

From orders table, find customer_ids who
have placed MORE than 2 orders.

Show : customer_id, total_orders
Order by total_orders descending.

--------------------------------------------------

Q3. WHERE + HAVING Together

From orders table :
- Use WHERE to keep only status = 'Delivered'
- Group by customer_id
- Use HAVING to keep only customers
  whose total delivered amount > 500

Show : customer_id, total_delivered_amount
Order by total_delivered_amount descending.

Think about it :
Why can't we put SUM(amount) > 500 in WHERE?

ALTER TABLE orders
ADD order_status VARCHAR2(20);

UPDATE orders
SET order_status =
    CASE
        WHEN MOD(order_id,5)=0 THEN 'Delivered'
        WHEN MOD(order_id,5)=1 THEN 'Pending'
        WHEN MOD(order_id,5)=2 THEN 'Cancelled'
        WHEN MOD(order_id,5)=3 THEN 'Returned'
        ELSE 'Shipped'
    END;

--------------------------------------------------

Q4. Department Active Headcount

From employees table, find departments that
have 3 or more ACTIVE employees (is_active = 1).

Show : department, active_count
Order by active_count descending.

--------------------------------------------------

Q5. HAVING with Multiple Conditions

From appointments table, find doctors who :
- Have seen MORE than 1 patient
- AND total fees collected > 600

Show : doctor_id, total_patients, total_fees
Order by total_fees descending.

--------------------------------------------------

PART B — SQL : JOINS
--------------------------------------------------

REFERENCE :
INNER JOIN     — only matching rows from both tables
LEFT JOIN      — all rows from left + NULLs for no match
RIGHT JOIN     — all rows from right + NULLs for no match
FULL OUTER JOIN — all rows from both sides
CROSS JOIN     — every row x every row (no condition)

--------------------------------------------------

Q6. INNER JOIN — Basics

Join customers and orders on customer_id.

Show : first_name, last_name, order_id,
       status, amount

After running, note down :
How many unique customers appear?
Does every customer in the customers table show up?
Why or why not?

--------------------------------------------------

Q7. LEFT JOIN — Spot the Difference

Run the same query from Q6 but with LEFT JOIN.

Note down :
Which customer now appears that was missing before?
What values does that customer have in order columns?
What does that NULL mean in real life?

--------------------------------------------------

Q8. Find Customers Who Never Ordered

(Classic Data Engineering interview question)

Use LEFT JOIN between customers and orders.
Show ONLY customers who have no orders at all.

Show : customer_id, first_name, last_name, _email_

Hint : After LEFT JOIN, filter where order_id IS NULL

--------------------------------------------------

Q9. INNER JOIN — Order + Product Details

Join orders and products on product_id.

Show : order_id, product_name, category,
       quantity, amount, status

Filter : only 'Delivered' orders
Order by amount descending.

--------------------------------------------------

Q10. 3-Table JOIN

Join orders + customers + products together.

Show :
- Full customer name (first_name + last_name)
  as customer_name
- product_name
- category
- quantity
- amount
- status

Order by amount descending.

--------------------------------------------------

Q11. LEFT JOIN + HAVING Combo

Join employees and department_info
on department column using LEFT JOIN.

Group by department.
Show departments where total salary bill > 200000.

Show : department, headcount, total_salary_bill
Order by total_salary_bill descending.

Note : Does the Legal department appear?
Why or why not?

--------------------------------------------------

Q12. FULL OUTER JOIN — Spot Missing Data

Join department_info and employees
using FULL OUTER JOIN on department.

Show : department, manager (from dept_info),
       employee name, salary

Observe and write down :
- Which department has no employees (NULL on emp side)?
- Which employee has no department info (NULL on dept side)?

--------------------------------------------------

Q13. Hospital — Multi Table JOIN

Join patients + appointments + doctors.

Show : patient name, doctor name, specialty,
       appointment date, diagnosis, fee

Filter : fee > 400
Order by fee descending.

BONUS :
Using GROUP BY + HAVING on the same join,
find doctors with total fees earned > 800.
Show : doctor name, total_fees

--------------------------------------------------

Q14. Recall Practice — NULL + WHERE + ORDER BY

From employees table :
- Show employees where salary IS NULL
  OR department IS NULL
- Also show employees with salary
  between 60000 and 80000
  who are active (is_active = 1)
- Order results by salary descending
- Use COALESCE to replace NULL salary with 0

(This question revises Day 9, 10, 11 concepts)

--------------------------------------------------

PART C — PYSPARK : NUMERIC FUNCTIONS
--------------------------------------------------

REFERENCE :
round(col, n)  — round to n decimal places
ceil(col)      — always round UP to next integer
floor(col)     — always round DOWN to next integer
abs(col)       — remove negative sign
sqrt(col)      — square root

--------------------------------------------------

Setup — Run this first :

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, round, ceil, floor, abs, sqrt,
    when, trim, upper, lower, initcap,
    to_date, year, month, datediff,
    current_date, concat, lit
)
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Day12_Practice") \
    .getOrCreate()

data = [
    (1, "  Alice Johnson  ", "Electronics",
     "2024-01-15", 1299.756, -120.5, 144.0),
    (2, "BOB SMITH",         "Electronics",
     "2024-02-20", 29.345,   -5.0,   25.0),
    (3, "carol williams",    "Furniture",
     "2024-01-10", 349.121,  -30.0,  81.0),
    (4, "DAVID BROWN",       "Furniture",
     "2024-03-05", 599.999,  -99.0,  49.0),
    (5, "  Eva Martinez  ",  "Electronics",
     "2024-02-28", 49.501,   -3.5,   36.0),
    (6, "Frank Lee",         "Stationery",
     "2024-01-22", 12.089,   0.0,    4.0),
    (7, "grace kim",         "Electronics",
     "2024-03-18", 129.499,  -15.0,  225.0),
    (8, "HENRY DAVIS",       "Furniture",
     "2024-02-14", 449.874,  -7.0,   196.0),
]

schema = StructType([
    StructField("sale_id",      IntegerType(), True),
    StructField("rep_name",     StringType(),  True),
    StructField("category",     StringType(),  True),
    StructField("sale_date_str",StringType(),  True),
    StructField("gross_amount", DoubleType(),  True),
    StructField("discount",     DoubleType(),  True),
    StructField("score_sq",     DoubleType(),  True),
])

df = spark.createDataFrame(data, schema)
df.show(truncate=False)
df.printSchema()

--------------------------------------------------

P1. round()

Add column gross_rounded = round(gross_amount, 2)
Show : rep_name, gross_amount, gross_rounded

--------------------------------------------------

P2. ceil() and floor()

Add two columns :
- price_ceil  = ceil(gross_amount)
- price_floor = floor(gross_amount)

Show : rep_name, gross_amount, price_ceil, price_floor

Observe : For 1299.756, what do ceil and floor return?
For 49.501, what do they return?

--------------------------------------------------

P3. abs() — Fix Negative Discounts

The discount column has negative values.
Use abs() to make them positive.

Add :
- discount_positive = abs(discount)
- net_amount = gross_amount - discount_positive

Show : rep_name, gross_amount,
       discount, discount_positive, net_amount

--------------------------------------------------

P4. sqrt() — Decode Scores

score_sq column has perfect square numbers.
Decode them using sqrt().

Add :
- score = round(sqrt(score_sq), 0)

Show : rep_name, score_sq, score

--------------------------------------------------

P5. Recall — String Functions

On the same df, clean the rep_name column :
- Remove leading/trailing spaces : trim()
- Convert to proper case : initcap()
- Store result in column : rep_name_clean

Show : rep_name (original), rep_name_clean

--------------------------------------------------

P6. Recall — Date Functions

Add these columns using sale_date_str :
- sale_date = to_date(sale_date_str, "yyyy-MM-dd")
- sale_year = year(sale_date)
- sale_month = month(sale_date)
- days_since_sale = datediff(current_date(), sale_date)

Show : rep_name_clean, sale_date,
       sale_year, sale_month, days_since_sale

--------------------------------------------------

P7. Full Transformation Pipeline

Starting from the original df,
apply ALL transformations in one chain :

Step 1 — String : trim + initcap on rep_name
         → rep_name_clean
Step 2 — Date   : to_date(sale_date_str)
         → sale_date
Step 3 — Date   : year, month, datediff
         → sale_year, sale_month, days_since_sale
Step 4 — Numeric : abs(discount)
         → discount_positive
Step 5 — Numeric : gross_amount - discount_positive
         → net_amount
Step 6 — Numeric : round(net_amount, 2)
         → net_rounded
Step 7 — Numeric : ceil(net_amount)
         → net_ceil
Step 8 — Numeric : round(sqrt(score_sq), 0)
         → score

Final select :
sale_id, rep_name_clean, category, sale_date,
sale_month, sale_year, days_since_sale,
gross_amount, discount_positive,
net_rounded, net_ceil, score

--------------------------------------------------

P8. Recall — Filtering + when()

Using the df from P7 :

Add a column net_tier using when() :
- net_amount < 50          -> "Low"
- net_amount 50 to 500     -> "Medium"
- net_amount above 500     -> "High"

Then filter : only Medium and High tier rows
Order by net_amount descending.

--------------------------------------------------

P9. Recall — groupBy + agg

Using the df from P7 :

Group by category.
Show :
- total_sales = count of rows
- total_gross = sum of gross_amount (round to 2)
- total_net = sum of net_amount (round to 2)
- avg_net = avg of net_amount (round to 2)
- avg_score = avg of score (round to 1)

Order by total_net descending.

--------------------------------------------------

P10. Recall — NULL Handling + dropDuplicates

Create this df :

null_data = [
    (1, "Alice",   "Electronics", 1200.5, -100.0),
    (2, "Bob",     None,           800.0,  -50.0),
    (3, None,      "Furniture",   3000.0, -200.0),
    (4, "Alice",   "Electronics", 1200.5, -100.0),
    (5, "Eva",     "Electronics",    0.0,    0.0),
    (6, "Frank",   "Stationery",  None,   -10.0),
]

Columns : id, rep_name, category,
          gross_amount, discount

Tasks :
- Show count of NULLs in each column
- Drop rows where rep_name is NULL
- Fill NULL in gross_amount with 0
- Fill NULL in category with "Unknown"
- Remove duplicate rows
- Show final clean df

--------------------------------------------------

MINI PROJECT — DAY 12
Sales Performance Report on GCP
--------------------------------------------------

Scenario :
You are a Data Engineer at a retail company.
Your manager wants a daily sales performance
report that combines data from multiple sources,
cleans it, and produces a final summary.

You will complete this entirely on GCP
using Google Cloud Shell or BigQuery.

--------------------------------------------------

PART 1 — SQL on BigQuery

Step 1 :
Go to BigQuery on your GCP account.
Create a dataset called : kodex_day12

Step 2 :
Create these two tables manually using
the BigQuery UI or SQL editor :

Table 1 : sales_reps
(rep_id, rep_name, region, is_active)

Insert at least 6 rows.
Include some reps with no sales (for JOIN practice).
Include one NULL in region.

Table 2 : daily_sales
(sale_id, rep_id, category, sale_date,
 gross_amount, discount, units_sold)

Insert at least 10 rows.
Include some reps from sales_reps.
Include one rep_id that does not exist
in sales_reps (for JOIN observation).

Step 3 : Run these queries in BigQuery

Query 1 :
LEFT JOIN sales_reps + daily_sales on rep_id.
Show all reps including those with no sales.
Identify reps with NULL sale data.

Query 2 :
INNER JOIN both tables.
Group by region.
Show : region, total_reps, total_sales,
       total_revenue, avg_revenue_per_sale.
Use HAVING to show only regions
with total_revenue > 500.

Query 3 :
Find the top performing rep per region.
Use GROUP BY rep_name, region.
Show : rep_name, region, total_revenue.
Filter : only active reps (is_active = 1).
Order by total_revenue descending.

--------------------------------------------------

PART 2 — PySpark on Cloud Shell

Step 1 :
Open GCP Cloud Shell.
Create a file called day12_project.py
using nano or vi.

Step 2 :
In the file, create the same sales data
as a PySpark DataFrame (hardcode the values,
no CSV needed for this project).

Columns : sale_id, rep_name, region,
          category, sale_date_str,
          gross_amount, discount, units_sold

At least 8 rows. Include messy rep_names
(mixed case, extra spaces) and negative discounts.

Step 3 :
Apply this full pipeline :

String Cleaning :
- trim + initcap on rep_name

Date Processing :
- Convert sale_date_str to date
- Extract year, month
- Calculate days_since_sale using datediff

Numeric Processing :
- discount_positive = abs(discount)
- net_amount = gross_amount - discount_positive
- net_rounded = round(net_amount, 2)
- revenue = round(net_amount * units_sold, 2)
- revenue_ceil = ceil(revenue)
- performance_score = round(sqrt(units_sold), 1)

Categorization using when() :
- revenue < 200  -> "Low Performer"
- revenue 200 to 1000 -> "Average Performer"
- revenue > 1000 -> "Top Performer"
Store in column : performance_label

NULL Handling :
- Fill any NULL in region with "Unknown"
- Drop rows where rep_name is NULL

Final Aggregation :
Group by region + performance_label.
Show : region, performance_label,
       rep_count, total_revenue,
       avg_performance_score
Order by total_revenue descending.

Step 4 :
Run the file using :
python3 day12_project.py

Take a screenshot of the final output.

-----------------------------------

SUBMISSION CHECKLIST

SQL :
- Q1 to Q5  : HAVING queries done
- Q6 to Q12 : JOIN queries done
- Q13, Q14  : Revision queries done
- Mini Project SQL : 3 BigQuery queries done

PySpark :
- P1 to P10 : All questions done
- Mini Project PySpark : Full pipeline done
  and run on Cloud Shell

--------------------------------

All the best.
KODE-X
'''

In [ ]:
# PART A — SQL : HAVING CLAUSE
# Q1
SELECT 
    DEPARTMENT,
    ROUND(AVG(SALARY), 2) AS AVG_SALARY
FROM EMPLOYEES
    GROUP BY DEPARTMENT
    HAVING AVG(SALARY) > 70000
    ORDER BY AVG_SALARY DESC;

#2
SELECT 
    CUSTOMER_ID,
    COUNT(*) AS TOTAL_ORDERS
FROM ORDERS
    GROUP BY CUSTOMER_ID
    HAVING COUNT(*) > 2
    ORDER BY TOTAL_ORDERS DESC;

#3
SELECT
    CUSTOMER_ID,
    SUM(TOTAL_AMOUNT) AS TOTAL_DELIVERED_AMOUNT
FROM ORDERS
WHERE ORDER_STATUS = 'Delivered'
GROUP BY CUSTOMER_ID
HAVING SUM(TOTAL_AMOUNT) > 500
ORDER BY TOTAL_DELIVERED_AMOUNT DESC;

#4
SELECT 
    DEPARTMENT,
    COUNT(*) AS ACTIVE_COUNT
FROM EMPLOYEES
WHERE IS_ACTIVE = 'YES'
GROUP BY DEPARTMENT
HAVING COUNT(*) >= 3
ORDER BY ACTIVE_COUNT DESC;

#5
SELECT
    DOCTOR_ID,
    COUNT(*) AS TOTAL_PATIENTS,
    SUM(FEE) AS TOTAL_FEES
FROM APPOINTMENTS
GROUP BY DOCTOR_ID
HAVING COUNT(*) > 1 AND SUM(FEE) > 600
ORDER BY TOTAL_FEES DESC;

In [ ]:
# PART B — SQL : JOINS
#6
SELECT 
    C.CUSTOMER_NAME,
    O.ORDER_ID,
    O.ORDER_STATUS,
    O.TOTAL_AMOUNT
FROM CUSTOMERS1 C
INNER JOIN ORDERS O
ON C.CUSTOMER_ID = O.CUSTOMER_ID;

'''
Questions Answer
1. How many unique customers appear?
Use:
SELECT COUNT(DISTINCT c.customer_id) AS unique_customers
FROM customers c
INNER JOIN orders o
ON c.customer_id = o.customer_id;

This counts unique customers who placed orders.

2. Does every customer in customers table show up?

No.

Only customers having matching orders appear.

3. Why?
Because INNER JOIN keeps only matching records between both tables.

If a customer has:
    no order
    or unmatched customer_id
then that customer is excluded.
'''
#7
SELECT 
    C.CUSTOMER_NAME,
    O.ORDER_ID,
    O.ORDER_STATUS,
    O.TOTAL_AMOUNT
FROM CUSTOMERS1 C
LEFT JOIN ORDERS O
ON C.CUSTOMER_ID = O.CUSTOMER_ID;

#8
SELECT 
    C.CUSTOMER_ID,
    C.CUSTOMER_NAME,
    C.EMAIL
FROM CUSTOMERS1 C
LEFT JOIN ORDERS O
ON C.CUSTOMER_ID = O.CUSTOMER_ID
WHERE O.ORDER_ID IS NULL;

#9
SELECT
    O.ORDER_ID,
    O.TOTAL_AMOUNT,
    O.ORDER_STATUS,
    O.QUANTITY,
    P.PRODUCT_NAME,
    P.CATEGORY
FROM ORDERS O
INNER JOIN PRODUCTS P
ON O.PRODUCT_ID = P.PRODUCT_ID
WHERE O.ORDER_STATUS = 'Delivered'
ORDER BY TOTAL_AMOUNT DESC;

#10 
SELECT 
    C.CUSTOMER_NAME,
    P.PRODUCT_NAME,
    P.CATEGORY,
    O.QUANTITY,
    O.TOTAL_AMOUNT,
    O.ORDER_STATUS
FROM ORDERS O
INNER JOIN CUSTOMERS1 C
ON O.CUSTOMER_ID = C.CUSTOMER_ID
INNER JOIN PRODUCTS P
ON O.PRODUCT_ID = P.PRODUCT_ID
ORDER BY O.TOTAL_AMOUNT DESC;

#11
SELECT 
    E.DEPARTMENT,
    COUNT(E.EMP_ID) AS HEADCOUNT,
    SUM(E.SALARY) AS TOTAL_SALARY_BILL
FROM EMPLOYEES E
LEFT JOIN DEPARTMENT_INFO1 D
ON E.DEPARTMENT = D.DEPARTMENT
GROUP BY E.DEPARTMENT
HAVING SUM(E.SALARY) > 200000
ORDER BY TOTAL_SALARY_BILL DESC;

#12
SELECT 
    D1.DEPARTMENT,
    D1.MANAGER_NAME,
    E1.EMP_NAME,
    E1.SALARY
FROM DEPARTMENT_INFO1 D1
FULL OUTER JOIN EMPLOYEES E1
ON D1.DEPARTMENT = E1.DEPARTMENT

'''
Observation 1
Which department has no employees?
    Admin
Why?
    Admin exists in department_info
    But no employee belongs to Admin
So employee columns become NULL.

Observation 2
Which employee has no department info?
    Anjali Nair
Why?
    Employee department = Support
    But Support department does not exist in department_info
So department columns become NULL.
'''

#13
SELECT 
    P.PATIENT_NAME,
    D.DOCTOR_NAME,
    D.SPECIALIZATION,
    A.APPOINTMENT_DATE,
    A.FEE
FROM APPOINTMENTS A
INNER JOIN PATIENTS P
ON A.PATIENT_ID = P.PATIENT_ID
INNER JOIN DOCTORS1 D
ON A.DOCTOR_ID = D.DOCTOR_ID
WHERE A.FEE > 400
ORDER BY A.FEE DESC;

# BONUS
SELECT 
    D.DOCTOR_NAME,
    SUM(A.FEE) AS TOTAL_FEES
FROM APPOINTMENTS A
INNER JOIN DOCTORS1 D
ON A.DOCTOR_ID = D.DOCTOR_ID
GROUP BY D.DOCTOR_NAME
HAVING SUM(A.FEE) > 800
ORDER BY TOTAL_FEES DESC;

#14
1. SELECT 
  EMP_NAME, 
  SALARY, 
  DEPARTMENT 
FROM EMPLOYEES 
WHERE SALARY IS NULL OR DEPARTMENT IS NULL;

2. SELECT 
  EMP_NAME, 
  SALARY 
  FROM EMPLOYEES 
  WHERE SALARY BETWEEN 60000 AND 80000 
  AND IS_ACTIVE = 'YES';

3. SELECT SALARY FROM EMPLOYEES ORDER BY SALARY DESC;

4. SELECT 
    EMP_ID,
    COALESCE(SALARY , 0) AS FINAL_SALARY 
    FROM EMPLOYEES


In [ ]:
# Note:
# pyspark answer is 'Google Colab Platform'
# mini project answer is 'GCP Cloud Platform' 